---
## CELL 1 — Mount Google Drive
Models save directly to your Drive — safe even if session restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os



SAVE_PATH = '/content/drive/MyDrive/nlp_trained_models'

os.makedirs(f'{SAVE_PATH}/tfidf_classifier',       exist_ok=True)
os.makedirs(f'{SAVE_PATH}/embedding_classifier',   exist_ok=True)
os.makedirs(f'{SAVE_PATH}/transformer_classifier', exist_ok=True)
os.makedirs(f'{SAVE_PATH}/chromadb',               exist_ok=True)

print('Drive mounted!')
print(f'Models will save to: {SAVE_PATH}')

Mounted at /content/drive
Drive mounted!
Models will save to: /content/drive/MyDrive/nlp_trained_models


---
## CELL 2 — Install Packages

In [ ]:
!pip install -q numpy==2.0.2
!pip install -q scikit-learn>=1.5.0
!pip install -q 'transformers>=4.40.0'
!pip install -q 'sentence-transformers>=3.0.0'
!pip install -q 'datasets>=2.20.0'
!pip install -q 'chromadb>=0.5.0'
!pip install -q nltk loguru

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('punkt',     quiet=True)

import numpy as np
import sklearn
import transformers
import torch

print('numpy       :', np.__version__)
print('sklearn     :', sklearn.__version__)
print('transformers:', transformers.__version__)
print('GPU         :', torch.cuda.is_available())
print('ALL PACKAGES READY')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

---
## CELL 3 — Load Dataset

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from collections import Counter
import numpy as np

CATEGORIES = [
    'sci.space',
    'sci.med',
    'sci.electronics',
    'comp.graphics',
    'comp.os.ms-windows.misc',
    'talk.politics.misc',
    'talk.religion.misc',
    'rec.sport.hockey',
    'rec.autos',
    'soc.religion.christian'
]

print('Loading 20 Newsgroups dataset...')

train_data = fetch_20newsgroups(
    subset='train', categories=CATEGORIES,
    remove=('headers','footers','quotes'), random_state=42
)
test_data = fetch_20newsgroups(
    subset='test', categories=CATEGORIES,
    remove=('headers','footers','quotes'), random_state=42
)

train_texts  = train_data.data
train_labels = [train_data.target_names[t] for t in train_data.target]
test_texts   = test_data.data
test_labels  = [test_data.target_names[t] for t in test_data.target]

print(f'Train : {len(train_texts)} samples')
print(f'Test  : {len(test_texts)} samples')
print(f'Classes: {CATEGORIES}')

Loading 20 Newsgroups dataset...
Train : 5588 samples
Test  : 3720 samples
Classes: ['sci.space', 'sci.med', 'sci.electronics', 'comp.graphics', 'comp.os.ms-windows.misc', 'talk.politics.misc', 'talk.religion.misc', 'rec.sport.hockey', 'rec.autos', 'soc.religion.christian']


---
## CELL 4 — Clean Text

In [ ]:
import re, string, unicodedata
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFKD', text).encode('ascii','ignore').decode('utf-8')
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.lower()
    text = text.translate(str.maketrans(string.punctuation, ' '*len(string.punctuation)))
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = []
    for tok in text.split():
        if len(tok) < 2: continue
        if tok.isdigit(): continue
        if tok in STOP_WORDS: continue
        tokens.append(LEMMATIZER.lemmatize(tok))
    return ' '.join(tokens)

print('Cleaning training texts...')
clean_train = [clean_text(t) for t in train_texts]
print('Cleaning test texts...')
clean_test  = [clean_text(t) for t in test_texts]

print('Done!')
print(f'Sample cleaned: {clean_train[0][:100]}')

Cleaning training texts...
Cleaning test texts...
Done!
Sample cleaned: ford automobile need information whether ford partially responsible car accident depletion ozone lay


---
## CELL 5 — Train TF-IDF + SVM (Baseline)

In [ ]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

print('='*50)
print('Training TF-IDF + SVM')
print('='*50)

tfidf = TfidfVectorizer(
    max_features=50000, ngram_range=(1,2),
    sublinear_tf=True, min_df=2, max_df=0.95
)
X_train_tfidf = tfidf.fit_transform(clean_train)
X_test_tfidf  = tfidf.transform(clean_test)
print(f'Feature matrix: {X_train_tfidf.shape}')

le = LabelEncoder()
y_train = le.fit_transform(train_labels)
y_test  = le.transform(test_labels)

print('Training SVM...')
svm = LinearSVC(C=1.0, max_iter=2000)
svm.fit(X_train_tfidf, y_train)

y_pred    = svm.predict(X_test_tfidf)
tfidf_acc = accuracy_score(y_test, y_pred)

print(f'\nTEST ACCURACY: {tfidf_acc:.4f} ({tfidf_acc*100:.2f}%)')
print(classification_report(y_test, y_pred, target_names=le.classes_))


with open(f'{SAVE_PATH}/tfidf_classifier/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open(f'{SAVE_PATH}/tfidf_classifier/svm_classifier.pkl', 'wb') as f:
    pickle.dump(svm, f)
with open(f'{SAVE_PATH}/tfidf_classifier/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('TF-IDF + SVM saved to Drive!')

Training TF-IDF + SVM
Feature matrix: (5588, 50000)
Training SVM...

TEST ACCURACY: 0.7559 (75.59%)
                         precision    recall  f1-score   support

          comp.graphics       0.78      0.80      0.79       389
comp.os.ms-windows.misc       0.80      0.75      0.78       394
              rec.autos       0.66      0.85      0.75       396
       rec.sport.hockey       0.93      0.92      0.92       399
        sci.electronics       0.76      0.70      0.73       393
                sci.med       0.82      0.79      0.81       396
              sci.space       0.77      0.75      0.76       394
 soc.religion.christian       0.71      0.84      0.77       398
     talk.politics.misc       0.73      0.58      0.65       310
     talk.religion.misc       0.48      0.39      0.43       251

               accuracy                           0.76      3720
              macro avg       0.75      0.74      0.74      3720
           weighted avg       0.76      0.76      0.7

---
## CELL 6 — Train Embeddings + KNN

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

print('='*50)
print('Training Dense Embeddings + KNN')
print('='*50)

print('Loading sentence-transformers model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print('Encoding training texts (~8 minutes)...')
train_embeddings = embedder.encode(
    train_texts, batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

print('Encoding test texts...')
test_embeddings = embedder.encode(
    test_texts, batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

y_train_emb = le.transform(train_labels)
y_test_emb  = le.transform(test_labels)

print('Training KNN...')
knn = KNeighborsClassifier(n_neighbors=7, metric='cosine', algorithm='brute', n_jobs=-1)
knn.fit(train_embeddings, y_train_emb)

y_pred_emb = knn.predict(test_embeddings)
emb_acc    = accuracy_score(y_test_emb, y_pred_emb)

print(f'\nTEST ACCURACY: {emb_acc:.4f} ({emb_acc*100:.2f}%)')
print(classification_report(y_test_emb, y_pred_emb, target_names=le.classes_))

with open(f'{SAVE_PATH}/embedding_classifier/knn_classifier.pkl', 'wb') as f:
    pickle.dump(knn, f)
with open(f'{SAVE_PATH}/embedding_classifier/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
np.save(f'{SAVE_PATH}/embedding_classifier/train_embeddings.npy', train_embeddings)

print('Embeddings + KNN saved to Drive!')

Training Dense Embeddings + KNN
Loading sentence-transformers model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

Encoding training texts (~8 minutes)...


Batches:   0%|          | 0/88 [00:00<?, ?it/s]

Encoding test texts...


Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Training KNN...

TEST ACCURACY: 0.7586 (75.86%)
                         precision    recall  f1-score   support

          comp.graphics       0.74      0.82      0.78       389
comp.os.ms-windows.misc       0.70      0.80      0.75       394
              rec.autos       0.82      0.84      0.83       396
       rec.sport.hockey       0.95      0.94      0.94       399
        sci.electronics       0.81      0.69      0.75       393
                sci.med       0.87      0.84      0.85       396
              sci.space       0.72      0.78      0.75       394
 soc.religion.christian       0.70      0.77      0.74       398
     talk.politics.misc       0.76      0.56      0.64       310
     talk.religion.misc       0.39      0.37      0.38       251

               accuracy                           0.76      3720
              macro avg       0.75      0.74      0.74      3720
           weighted avg       0.76      0.76      0.76      3720

Embeddings + KNN saved to Drive!


---
## CELL 7 — Fine-tune RoBERTa

> Make sure GPU is ON: Runtime → Change runtime type → T4 GPU

In [ ]:
import torch, json
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')

print('='*50)
print('Fine-tuning RoBERTa')
print('='*50)

classes  = list(le.classes_)
id2label = {i: c for i, c in enumerate(classes)}
label2id = {c: i for i, c in enumerate(classes)}

y_train_rob = le.transform(train_labels)
y_test_rob  = le.transform(test_labels)

tr_texts, val_texts, tr_labels_r, val_labels_r = train_test_split(
    train_texts, y_train_rob, test_size=0.1, random_state=42, stratify=y_train_rob
)

MODEL_NAME = 'roberta-base'
print(f'Loading tokenizer: {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts, labels):
    enc = tokenizer(texts, truncation=True, max_length=256, padding='max_length')
    return Dataset.from_dict({
        'input_ids':      enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'labels':         list(labels)
    })

print('Tokenizing...')
train_dataset = tokenize(tr_texts,   tr_labels_r)
val_dataset   = tokenize(val_texts,  val_labels_r)
test_dataset  = tokenize(test_texts, y_test_rob)

print('Loading RoBERTa model...')
rob_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(classes), id2label=id2label, label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir=f'{SAVE_PATH}/transformer_classifier',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

trainer = Trainer(
    model=rob_model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print('Training... (15-20 minutes)')
trainer.train()

print('Evaluating on test set...')
test_results = trainer.predict(test_dataset)
test_preds   = np.argmax(test_results.predictions, axis=-1)
rob_acc      = accuracy_score(y_test_rob, test_preds)

print(f'\nTEST ACCURACY: {rob_acc:.4f} ({rob_acc*100:.2f}%)')
print(classification_report(y_test_rob, test_preds, target_names=classes))

rob_model.save_pretrained(f'{SAVE_PATH}/transformer_classifier')
tokenizer.save_pretrained(f'{SAVE_PATH}/transformer_classifier')
with open(f'{SAVE_PATH}/transformer_classifier/label_mappings.json', 'w') as f:
    json.dump({'id2label': {str(k): v for k, v in id2label.items()}, 'label2id': label2id}, f, indent=2)

print('RoBERTa saved to Drive!')

GPU: True
GPU name: Tesla T4
Fine-tuning RoBERTa
Loading tokenizer: roberta-base...
Tokenizing...
Loading RoBERTa model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training... (15-20 minutes)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.804431,0.675050,0.763864
2,0.565981,0.606284,0.787120
3,0.364520,0.574102,0.817531
4,0.218286,0.605468,0.822898


...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Evaluating on test set...



TEST ACCURACY: 0.7712 (77.12%)
                         precision    recall  f1-score   support

          comp.graphics       0.76      0.80      0.78       389
comp.os.ms-windows.misc       0.77      0.79      0.78       394
              rec.autos       0.70      0.85      0.77       396
       rec.sport.hockey       0.95      0.92      0.94       399
        sci.electronics       0.81      0.71      0.76       393
                sci.med       0.88      0.83      0.85       396
              sci.space       0.82      0.78      0.80       394
 soc.religion.christian       0.76      0.82      0.79       398
     talk.politics.misc       0.73      0.60      0.66       310
     talk.religion.misc       0.43      0.45      0.44       251

               accuracy                           0.77      3720
              macro avg       0.76      0.76      0.76      3720
           weighted avg       0.78      0.77      0.77      3720



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa saved to Drive!


---
## CELL 8 — Save Final Results

In [ ]:
import pandas as pd

baseline_err = 1 - tfidf_acc
best_err     = 1 - rob_acc
reduction    = (baseline_err - best_err) / baseline_err * 100

print('='*50)
print('   FINAL RESULTS')
print('='*50)
print(f'TF-IDF + SVM     : {tfidf_acc*100:.2f}%')
print(f'Embeddings + KNN : {emb_acc*100:.2f}%')
print(f'RoBERTa          : {rob_acc*100:.2f}%')
print(f'Misclassification reduction: {reduction:.1f}%')
print('='*50)

with open(f'{SAVE_PATH}/final_accuracy.txt', 'w') as f:
    f.write(f'TF-IDF + SVM Accuracy      : {tfidf_acc*100:.2f}%\n')
    f.write(f'Embedding + KNN Accuracy   : {emb_acc*100:.2f}%\n')
    f.write(f'RoBERTa Accuracy           : {rob_acc*100:.2f}%\n')
    f.write(f'Misclassification Reduction: {reduction:.1f}%\n')

pd.DataFrame([
    {'Model': 'TF-IDF + SVM',      'Accuracy': f'{tfidf_acc*100:.2f}%'},
    {'Model': 'Embeddings + KNN',  'Accuracy': f'{emb_acc*100:.2f}%'},
    {'Model': 'Fine-tuned RoBERTa','Accuracy': f'{rob_acc*100:.2f}%'},
]).to_csv(f'{SAVE_PATH}/training_results.csv', index=False)

print('Results saved to Drive!')

   FINAL RESULTS
TF-IDF + SVM     : 75.59%
Embeddings + KNN : 75.86%
RoBERTa          : 77.12%
Misclassification reduction: 6.3%
Results saved to Drive!


---
## CELL 9 — Index ChromaDB

In [ ]:
import chromadb, uuid

CHROMA_PATH = f'{SAVE_PATH}/chromadb'
print('Setting up ChromaDB...')

client = chromadb.PersistentClient(path=CHROMA_PATH)
try:
    client.delete_collection('document_store')
except:
    pass

collection = client.get_or_create_collection(
    name='document_store',
    metadata={'hnsw:space': 'cosine'}
)

INDEX_COUNT = min(2000, len(train_texts))
BATCH_SIZE  = 100
print(f'Indexing {INDEX_COUNT} documents...')

for start in range(0, INDEX_COUNT, BATCH_SIZE):
    end       = min(start + BATCH_SIZE, INDEX_COUNT)
    batch_txt = train_texts[start:end]
    batch_lbl = train_labels[start:end]

    embeddings = embedder.encode(
        batch_txt,
        normalize_embeddings=True,
        show_progress_bar=False
    ).tolist()

    collection.add(
        documents=batch_txt,
        embeddings=embeddings,
        metadatas=[{'label': l} for l in batch_lbl],
        ids=[str(uuid.uuid4()) for _ in batch_txt]
    )
    if (start // BATCH_SIZE) % 5 == 0:
        print(f'  Indexed {end}/{INDEX_COUNT}...')

print(f'Total indexed: {collection.count()}')

q_emb   = embedder.encode(['rocket launch space mission'], normalize_embeddings=True).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)
print('\nSearch test: rocket launch space mission')
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f'  [{meta["label"]}] {doc[:70]}...')

print('\nChromaDB indexed and saved to Drive!')

Setting up ChromaDB...
Indexing 2000 documents...
  Indexed 100/2000...
  Indexed 600/2000...
  Indexed 1100/2000...
  Indexed 1600/2000...
Total indexed: 2000

Search test: rocket launch space mission
  [sci.space] To All -- I thought the net would find this amusing..
  
From the Marc...
  [sci.space] For an essay, I am writing about the space shuttle and a need for a be...
  [sci.space] Joe,

	your description sounds like one of the  gravity probe  spacecr...

ChromaDB indexed and saved to Drive!


---
## CELL 10 — Test Predictions (verify everything works)

In [ ]:
import torch

test_sentences = [
    'NASA launches new satellite to study climate change from orbit',
    'Stock market crashes after Federal Reserve raises interest rates',
    'Scientists develop new drug that targets cancer cells directly',
    'Python 3.13 released with new JIT compiler for faster execution',
    'Manchester United wins championship after dramatic penalty shootout',
]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
rob_model.to(device)
rob_model.eval()

print('='*60)
print('   LIVE PREDICTION TEST')
print('='*60)

for sentence in test_sentences:

    vec        = tfidf.transform([clean_text(sentence)])
    tfidf_pred = le.inverse_transform(svm.predict(vec))[0]


    inputs = tokenizer(
        sentence, return_tensors='pt',
        truncation=True, max_length=256, padding=True
    ).to(device)

    with torch.no_grad():
        outputs = rob_model(**inputs)
        probs   = torch.softmax(outputs.logits, dim=-1).squeeze()
        pred_id = probs.argmax().item()
        conf    = probs[pred_id].item()
    rob_pred = id2label[pred_id]

    print(f'\nText    : {sentence[:55]}...')
    print(f'TF-IDF  : {tfidf_pred}')
    print(f'RoBERTa : {rob_pred} ({conf:.1%})')

print('\nAll predictions working!')

   LIVE PREDICTION TEST

Text    : NASA launches new satellite to study climate change fro...
TF-IDF  : sci.space
RoBERTa : sci.space (98.8%)

Text    : Stock market crashes after Federal Reserve raises inter...
TF-IDF  : talk.politics.misc
RoBERTa : talk.politics.misc (92.8%)

Text    : Scientists develop new drug that targets cancer cells d...
TF-IDF  : sci.med
RoBERTa : sci.med (98.7%)

Text    : Python 3.13 released with new JIT compiler for faster e...
TF-IDF  : rec.autos
RoBERTa : comp.os.ms-windows.misc (81.9%)

Text    : Manchester United wins championship after dramatic pena...
TF-IDF  : rec.sport.hockey
RoBERTa : rec.sport.hockey (98.2%)

All predictions working!


---
## CELL 11 — Download Everything

In [ ]:
import shutil
from google.colab import files

print('Zipping all models from Drive...')
shutil.make_archive('/content/trained_models', 'zip', SAVE_PATH)

print('Starting download...')
files.download('/content/trained_models.zip')

print('Done! Check your Downloads folder for trained_models.zip')
print(f'\nYour accuracy numbers:')

with open(f'{SAVE_PATH}/final_accuracy.txt') as f:
    print(f.read())

Zipping all models from Drive...
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Check your Downloads folder for trained_models.zip

Your accuracy numbers:
TF-IDF + SVM Accuracy      : 75.59%
Embedding + KNN Accuracy   : 75.86%
RoBERTa Accuracy           : 77.12%
Misclassification Reduction: 6.3%

